In [ ]:
import os
os.environ['JAX_ENABLE_X64'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1'

%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

import jax
import jax.numpy as jnp

import numpy as np
from temgym_core.aberrations import KrivanekCoeffs
from temgym_core.components import Detector
from temgym_core.components import KrivanekLens, PhaseBiprism as Biprism
from temgym_core.run import run_to_end_vmapped
from temgym_core.source import square_input_wave

from temgym_core.plotting import plot_model, PlotParams

from temgym_core.constants import energy2wavelength
from temgym_core.transfer_matrices import calculate_z1_and_z2_from_M_and_f
from temgym_core.evaluate import evaluate_gaussians_gpu_kernel_wrapper

jax.config.update("jax_enable_x64", True)


Inward biprism condition
========================

For this notebook the biprism coordinate is aligned with `x`, so the phase action is
`S_bi(x) = -strength * abs(x)`. The added ray slope is therefore
`dtheta_x = dS_bi / dx = -strength * sign(x)`. A positive `strength` kicks the
`x > 0` half toward negative `x` and the `x < 0` half toward positive `x`, i.e. inward.

To make that inward kick visually line up with the objective ray fan, the biprism is placed
before the objective crossover: `lens.z < biprism.z < lens.z + lens.focal_length`.


Simulation parameters
=====================

In [ ]:
W = 500e-9
voltage = 200e3
wavelength = energy2wavelength(voltage)
k0 = 2 * jnp.pi / wavelength

Nx = Ny = 1024
dx = W / Nx
dy = W / Ny

obj_mag = -20
obj_focal_length = 1e-3
z1, z2 = calculate_z1_and_z2_from_M_and_f(obj_mag, obj_focal_length)
z1, z2 = abs(z1), abs(z2)
image_z = z1 + z2
objective_focus_z = z1 + obj_focal_length

biprism_fraction_to_focus = 0.45
biprism_z = z1 + biprism_fraction_to_focus * obj_focal_length
assert z1 < biprism_z < objective_focus_z

fringe_spacing = 40e-9
b_def = wavelength / (2 * fringe_spacing)

print(f"objective lens z     = {z1:.6e} m")
print(f"objective focus z    = {objective_focus_z:.6e} m")
print(f"biprism z            = {biprism_z:.6e} m")
print(f"image plane z        = {image_z:.6e} m")
print(f"biprism strength     = {float(b_def):.6e} rad")
print(f"carrier fringe pitch = {fringe_spacing:.6e} m")


Model Creation
==============

In [ ]:
input_grid = Detector(z=0.0, pixel_size=(dx, dy), shape=(Nx, Ny))
input_extent = input_grid.extent

output_fov = abs(obj_mag) * W
output_dx = output_fov / Nx
output_dy = output_fov / Ny
output_grid = Detector(z=image_z, pixel_size=(output_dx, output_dy), shape=(Nx, Ny))
output_extent = output_grid.extent
xy = output_grid.coords_1d
x = xy[0]
y = xy[1]

# Keep the aberration visible but smaller than the objective convergence.
coma_coeffs = KrivanekCoeffs(
    C21=0.25,
    phi21=0.0,
    C12=5.0e-5,
    phi12=jnp.pi / 4,
    C30=100.0,
)
lens = KrivanekLens(z=z1, focal_length=obj_focal_length, coeffs=coma_coeffs)
biprism = Biprism(z=biprism_z, strength=b_def, width=1e-12, sharpness=1e12)

print(f"Output field of view = {output_fov:.3e} m")
print(f"Output pixel size    = {output_dx:.3e} m")
print(f"Using KrivanekLens with C21 coma = {coma_coeffs.C21} m")


Lens aberration phase
======================


In [ ]:
phase_stride = max(Nx // 512, 1)
phase_x = input_grid.coords_1d[0][::phase_stride]
phase_y = input_grid.coords_1d[1][::phase_stride]
phase_X, phase_Y = jnp.meshgrid(phase_x, phase_y, indexing="xy")
lens_coords = jnp.stack((phase_X.ravel(), phase_Y.ravel()), axis=-1)
phase_shape = phase_X.shape
phase_extent = (phase_x[0], phase_x[-1], phase_y[0], phase_y[-1])

ideal_lens = KrivanekLens(
    z=z1,
    focal_length=obj_focal_length,
    coeffs=KrivanekCoeffs(),
)
phase_ideal = jax.vmap(ideal_lens.phase_shift)(lens_coords).reshape(phase_shape)
phase_aberrated = jax.vmap(lens.phase_shift)(lens_coords).reshape(phase_shape)
aberration_phase = np.asarray((phase_aberrated - phase_ideal) * k0)

fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(
    np.mod(aberration_phase + np.pi, 2 * np.pi) - np.pi,
    extent=phase_extent,
    origin="lower",
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi,
)
ax.set_title("Lens phase error")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.axis("equal")
fig.colorbar(im, ax=ax, label="wrapped phase error (rad)")


Input Wavefront
================

In [ ]:
rays_in = square_input_wave(aperture_length=W,
                            waist=4e-9,
                            voltage=voltage,
                            amp=1.0,
                            phase=0.0,
                            overlap_factor=1.5,
                            z0=0.0)

print(f"Number of rays: {len(rays_in.x)}")

Sum gaussians and visualise input wavefront
=======================================================

In [ ]:
E_in = evaluate_gaussians_gpu_kernel_wrapper(rays_in, input_grid).block_until_ready()

fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(np.abs(E_in), extent=input_extent, origin="lower")
ax.set_title("Input amplitude")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.axis("equal")
fig.colorbar(im, ax=ax)


See aberrated biprism hologram
================================

In [ ]:
ray_out = run_to_end_vmapped(rays_in, [lens, biprism, output_grid])
E_out = evaluate_gaussians_gpu_kernel_wrapper(ray_out, output_grid).block_until_ready()

In [ ]:
intensity_out = np.abs(E_out) ** 2
phase_out = np.angle(E_out)

fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)

im0 = axs[0].imshow(np.abs(E_out), extent=output_extent, origin="lower")
axs[0].set_title("Amplitude")
axs[0].set_xlabel("x (m)")
axs[0].set_ylabel("y (m)")
axs[0].axis("equal")
fig.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].imshow(
    intensity_out,
    extent=output_extent,
    origin="lower",
    cmap="magma",
    norm=PowerNorm(gamma=0.35),
)
axs[1].set_title("Biprism hologram")
axs[1].set_xlabel("x (m)")
axs[1].axis("equal")
fig.colorbar(im1, ax=axs[1], fraction=0.046)

im2 = axs[2].imshow(
    phase_out,
    extent=output_extent,
    origin="lower",
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi,
)
axs[2].set_title("Wrapped phase")
axs[2].set_xlabel("x (m)")
axs[2].axis("equal")
fig.colorbar(im2, ax=axs[2], fraction=0.046)


Verify that the output fringe spacing matches theoretical value
=======================================================

In [ ]:
# Measure fringe spacing from the central row of the hologram.
intensity = np.abs(E_out[Nx // 2, :]) ** 2
x_np = np.array(x)
int_np = np.array(intensity)

# Peak detection: local maxima above threshold (try 20% of max, then 5% if too few).
th = int_np.max() * 0.20
peaks = np.where((int_np[1:-1] > int_np[:-2]) & (int_np[1:-1] > int_np[2:]) & (int_np[1:-1] > th))[0] + 1
if len(peaks) < 2:
    th = int_np.max() * 0.05
    peaks = np.where((int_np[1:-1] > int_np[:-2]) & (int_np[1:-1] > int_np[2:]) & (int_np[1:-1] > th))[0] + 1

if len(peaks) >= 2:
    spacings = np.diff(x_np[peaks])
    mean_spacing = spacings.mean()
    std_spacing = spacings.std()
    print(f"Detected {len(peaks)} peaks. Mean fringe spacing = {mean_spacing:.3e} m (std {std_spacing:.3e})")
    print(f"Requested carrier spacing = {fringe_spacing:.3e} m -> ratio measured/requested = {mean_spacing / fringe_spacing:.3f}")
else:
    print("Could not detect enough peaks to measure spacing.")

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.plot(x_np, int_np, label="Intensity (central row)")
if len(peaks) > 0:
    ax.plot(x_np[peaks], int_np[peaks], "r.", label="Peaks")
ax.set_xlabel("x (m)")
ax.set_ylabel("Intensity")
ax.set_title("Central hologram fringe profile")
ax.legend()


View the aberrated ray paths
==============================

In [ ]:
plot_stride = max(len(rays_in.x) // 500, 1)
rays_for_plot = rays_in[0::plot_stride].derive(voltage=voltage)

fig, ax = plot_model(
    components=[input_grid, lens, biprism, output_grid],
    rays=rays_for_plot,
    ray_coordinate="r",
    r_side_by_sign=True,
    plot_params=PlotParams(
        figsize=(10, 6),
        ray_lw=0.9,
        ray_alpha=0.55,
        lens_height=1e-6,
        biprism_radius=2e-7,
    ),
)
ax.axhline(objective_focus_z, color="0.4", lw=0.8, ls="--")
ax.text(0.98, objective_focus_z, "objective focus", transform=ax.get_yaxis_transform(), ha="right", va="bottom", color="0.3")
ax.set_title("Lens and inward-kicking biprism ray fan")
